# AutoLM -- Pretraining auf Google Colab

**Dieses Notebook enthaelt bewusst KEINE eigene Logik.** Alle Ideen und Algorithmen
stehen im Repo (`kern/`, `daten/`) -- ein Notebook ist in einem Diff nicht sauber
pruefbar, ein Python-Skript schon. Dieses Notebook tut nur vier Dinge:

1. Repo klonen
2. Google Drive einhaengen (fuer absturzsichere Checkpoints -- siehe `kern/checkpoint.py`)
3. Daten vortokenisieren, falls noch nicht auf Drive vorhanden
4. `kern/trainiere.py` aufrufen

**Vor dem ersten Lauf:** Laufzeit -> Laufzeittyp aendern -> GPU (T4 reicht).

In [ ]:
# 1. Repo klonen
!git clone https://github.com/Atakan-24/autolm.git
%cd autolm
!pip install -q torch numpy pyarrow

In [ ]:
# 2. Google Drive einhaengen -- HIER liegen die Checkpoints, nicht lokal.
# Grund (siehe kern/checkpoint.py): eine Colab-Sitzung kann jederzeit ohne
# Vorwarnung enden. Was nur lokal in der Colab-VM liegt, ist dann weg.
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_ORDNER = '/content/drive/MyDrive/autolm-checkpoints'
DATEN_ORDNER = '/content/drive/MyDrive/autolm-daten'

import os
os.makedirs(CHECKPOINT_ORDNER, exist_ok=True)
os.makedirs(DATEN_ORDNER, exist_ok=True)

In [ ]:
# 3. Daten vortokenisieren -- NUR falls noch nicht auf Drive vorhanden.
# Bei einem zweiten/dritten Colab-Lauf (nach einem Abbruch) ist dieser
# Schritt ein No-Op: vortokenisiere.py prueft selbst, welche Shards schon
# verarbeitet sind (daten/tinystories_meta.json), und laedt/verarbeitet
# nur, was fehlt.
!ln -sf {DATEN_ORDNER} daten_persistent
!python daten/vortokenisiere.py --hoechstens-token 340000000 || true

# Ergebnis auf Drive spiegeln, damit ein KUENFTIGER Colab-Lauf es wiederfindet
!cp daten/tinystories_train.bin daten/tinystories_meta.json daten/tokenizer.pkl {DATEN_ORDNER}/ 2>/dev/null || true

In [ ]:
# Falls die Daten schon von einem FRUEHEREN Lauf auf Drive liegen: von dort
# zurueckkopieren statt neu herunterzuladen/tokenisieren.
!cp {DATEN_ORDNER}/tinystories_train.bin daten/ 2>/dev/null || true
!cp {DATEN_ORDNER}/tinystories_meta.json daten/ 2>/dev/null || true
!cp {DATEN_ORDNER}/tokenizer.pkl daten/ 2>/dev/null || true

!ls -lh daten/*.bin daten/*.json 2>/dev/null

In [ ]:
# 4. Training. --gross = die 17M-Konfiguration aus dem Plan.
# Bricht die Sitzung ab: dieselbe Zelle einfach nochmal ausfuehren --
# kern/checkpoint.py findet den letzten Checkpoint auf Drive und macht
# genau dort weiter (bewiesen in kern/test_checkpoint.py, echter
# Kill-und-Resume-Test, keine Simulation).
!python kern/trainiere.py \
    --daten daten/tinystories_train.bin \
    --meta daten/tinystories_meta.json \
    --checkpoint-ordner {CHECKPOINT_ORDNER} \
    --gross \
    --token-budget 340000000

## Nach dem Training

Verlust-Log liegt unter `{CHECKPOINT_ORDNER}/ckpt_verlust.jsonl` -- eine Zeile pro
Checkpoint, `{"schritt": ..., "verlust": ...}`. Fuer die Loss-Kurve im README:
lokal herunterladen und mit einem beliebigen Plot-Werkzeug darstellen (kein
Notebook-Code hier -- das gehoert als eigenes, pruefbares Skript ins Repo,
sobald das Training durchgelaufen ist).